# 🌪️ EDA — NLP with Disaster Tweets (Kaggle)

**Objectif** : Explorer le dataset Kaggle *Natural Language Processing with Disaster Tweets* pour comprendre les données avant modélisation.

### Plan :
0. Configuration & Imports
1. Chargement & Inspection rapide
2. Valeurs manquantes
3. Analyse de la variable cible
4. Prétraitement du texte
5. Keyword & Location
6. Features textuelles
7. Analyse textuelle (fréquences, diversity, longueur)
8. WordClouds
9. N-grammes
10. TF-IDF — mots discriminants
11. Hashtags / Mentions / URLs
12. Doublons & Labels conflictuels
13. Train vs Test
14. Longueur de séquence (préparation modèles)
15. Exemples de tweets par classe
16. Résumé & Insights
17. Sauvegarde des données

## 0. Configuration & Imports

In [ ]:
# Installation des dépendances (à exécuter une seule fois)
!pip install emoji -q

In [ ]:
import re
import warnings
import string
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import emoji

import nltk
for resource in ['stopwords', 'punkt', 'punkt_tab', 'wordnet', 'averaged_perceptron_tagger']:
    nltk.download(resource, quiet=True)

from nltk.corpus import stopwords
from nltk.util import ngrams

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

from wordcloud import WordCloud

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 120)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

RANDOM_STATE = 42
RAW_PATH     = 'tweets.csv'  # Adjust if needed

print('✅ Librairies importées avec succès')

## 1. Chargement & Inspection rapide

In [ ]:
# Chargement
df = pd.read_csv(RAW_PATH, engine='python', on_bad_lines='skip', sep=';')
print(f'Dataset brut : {df.shape}')

# Supprimer les colonnes entièrement vides (e.g., 'Unnamed: X')
df.dropna(axis=1, how='all', inplace=True)
unnamed_cols = [col for col in df.columns if 'Unnamed:' in col]
if unnamed_cols:
    df.drop(columns=unnamed_cols, inplace=True)
print(f'Dataset après suppression des colonnes vides : {df.shape}')

In [ ]:
print('=== .info() ===')
df.info()

In [ ]:
print('=== .describe(include=all) ===')
df.describe(include='all')

In [ ]:
print('=== Premier tweet brut ===')
print(df['text'].iloc[0])

## 2. Valeurs manquantes

In [ ]:
def missing_summary(dataframe, name):
    missing     = dataframe.isnull().sum()
    missing_pct = (missing / len(dataframe)) * 100
    summary = pd.DataFrame({'Missing': missing, 'Missing (%)': missing_pct.round(2)})
    print(f'\n=== Valeurs manquantes — {name} ===')
    return summary[summary['Missing'] > 0]

display(missing_summary(df, 'Dataset complet'))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
mp = df.isnull().mean() * 100
mp = mp[mp > 0]
if len(mp) > 0:
    ax.barh(mp.index, mp.values, color='#FF6B6B', edgecolor='black')
    ax.set_xlabel('% manquant')
    ax.set_title('Valeurs manquantes — Dataset complet', fontsize=13, fontweight='bold')
    for i, v in enumerate(mp.values):
        ax.text(v + 0.3, i, f'{v:.1f}%', va='center')
else:
    ax.text(0.5, 0.5, 'Aucune valeur manquante', ha='center', va='center', transform=ax.transAxes)
plt.tight_layout()
plt.show()

## 3. Analyse de la variable cible (`target`)

In [ ]:
# Conversion de la cible
df['target'] = pd.to_numeric(df['target'], errors='coerce')
if df['target'].isnull().any():
    print(f"NaN values found in 'target'. Dropping {df['target'].isnull().sum()} rows.")
    df.dropna(subset=['target'], inplace=True)
df['target'] = df['target'].astype(int)

target_counts = df['target'].value_counts()
target_pct    = df['target'].value_counts(normalize=True) * 100

print('Distribution de la variable target :')
print(pd.DataFrame({'Count': target_counts, 'Percentage (%)': target_pct.round(2)}))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

labels = ['Non-Disaster (0)', 'Disaster (1)']
colors = ['#4ECDC4', '#FF6B6B']

axes[0].bar(labels, target_counts.values, color=colors, edgecolor='black', width=0.5)
axes[0].set_title('Distribution de la cible', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Nombre de tweets')
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold')

axes[1].pie(target_counts.values, labels=labels, autopct='%1.1f%%',
            colors=colors, startangle=140,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Part de chaque classe', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Prétraitement du texte

### 4.1 Définition des fonctions de nettoyage

In [ ]:
STOPWORDS_EN = set(stopwords.words('english'))

def clean(tweet):
    """Nettoyage cas par cas : emojis, contractions, caractères spéciaux, hashtags nommés, typos."""

    # ── NOUVELLES ÉTAPES ────────────────────────────────────────────────────
    # 1. Conversion des emojis en description textuelle
    tweet = emoji.demojize(tweet, delimiters=(' ', ' '))

    # 2. Suppression des URLs (version élargie)
    tweet = re.sub(r'https?://\S+', '', tweet)
    tweet = re.sub(r'www\.\S+', '', tweet)

    # 3. Extraction du texte des hashtags (ex: #LondonFire → LondonFire)
    tweet = re.sub(r'#(\w+)', r'\1', tweet)

    # 4. Suppression des mentions (@username)
    tweet = re.sub(r'@\w+', '', tweet)

    # ── CARACTÈRES SPÉCIAUX (encodage) ──────────────────────────────────────
    tweet = re.sub(r"\x89Û_", "", tweet)
    tweet = re.sub(r"\x89ÛÒ", "", tweet)
    tweet = re.sub(r"\x89ÛÓ", "", tweet)
    tweet = re.sub(r"\x89ÛÏWhen", "When", tweet)
    tweet = re.sub(r"\x89ÛÏ", "", tweet)
    tweet = re.sub(r"China\x89Ûªs", "China's", tweet)
    tweet = re.sub(r"let\x89Ûªs", "let's", tweet)
    tweet = re.sub(r"\x89Û÷", "", tweet)
    tweet = re.sub(r"\x89Ûª", "", tweet)
    tweet = re.sub(r"\x89Û\x9d", "", tweet)
    tweet = re.sub(r"å_", "", tweet)
    tweet = re.sub(r"\x89Û¢", "", tweet)
    tweet = re.sub(r"\x89Û¢åÊ", "", tweet)
    tweet = re.sub(r"fromåÊwounds", "from wounds", tweet)
    tweet = re.sub(r"åÊ", "", tweet)
    tweet = re.sub(r"åÈ", "", tweet)
    tweet = re.sub(r"JapÌ_n", "Japan", tweet)
    tweet = re.sub(r"Ì©", "e", tweet)
    tweet = re.sub(r"å¨", "", tweet)
    tweet = re.sub(r"SuruÌ¤", "Suruc", tweet)
    tweet = re.sub(r"åÇ", "", tweet)
    tweet = re.sub(r"å£3million", "3 million", tweet)
    tweet = re.sub(r"åÀ", "", tweet)

    # ── CONTRACTIONS ────────────────────────────────────────────────────────
    tweet = re.sub(r"he's", "he is", tweet)
    tweet = re.sub(r"there's", "there is", tweet)
    tweet = re.sub(r"We're", "We are", tweet)
    tweet = re.sub(r"That's", "That is", tweet)
    tweet = re.sub(r"won't", "will not", tweet)
    tweet = re.sub(r"they're", "they are", tweet)
    tweet = re.sub(r"Can't", "Cannot", tweet)
    tweet = re.sub(r"wasn't", "was not", tweet)
    tweet = re.sub(r"don\x89Ûªt", "do not", tweet)
    tweet = re.sub(r"aren't", "are not", tweet)
    tweet = re.sub(r"isn't", "is not", tweet)
    tweet = re.sub(r"What's", "What is", tweet)
    tweet = re.sub(r"haven't", "have not", tweet)
    tweet = re.sub(r"hasn't", "has not", tweet)
    tweet = re.sub(r"There's", "There is", tweet)
    tweet = re.sub(r"He's", "He is", tweet)
    tweet = re.sub(r"It's", "It is", tweet)
    tweet = re.sub(r"You're", "You are", tweet)
    tweet = re.sub(r"I'M", "I am", tweet)
    tweet = re.sub(r"shouldn't", "should not", tweet)
    tweet = re.sub(r"wouldn't", "would not", tweet)
    tweet = re.sub(r"i'm", "I am", tweet)
    tweet = re.sub(r"I\x89Ûªm", "I am", tweet)
    tweet = re.sub(r"I'm", "I am", tweet)
    tweet = re.sub(r"Isn't", "is not", tweet)
    tweet = re.sub(r"Here's", "Here is", tweet)
    tweet = re.sub(r"you've", "you have", tweet)
    tweet = re.sub(r"you\x89Ûªve", "you have", tweet)
    tweet = re.sub(r"we're", "we are", tweet)
    tweet = re.sub(r"what's", "what is", tweet)
    tweet = re.sub(r"couldn't", "could not", tweet)
    tweet = re.sub(r"we've", "we have", tweet)
    tweet = re.sub(r"it\x89Ûªs", "it is", tweet)
    tweet = re.sub(r"doesn\x89Ûªt", "does not", tweet)
    tweet = re.sub(r"It\x89Ûªs", "It is", tweet)
    tweet = re.sub(r"Here\x89Ûªs", "Here is", tweet)
    tweet = re.sub(r"who's", "who is", tweet)
    tweet = re.sub(r"I\x89Ûªve", "I have", tweet)
    tweet = re.sub(r"y'all", "you all", tweet)
    tweet = re.sub(r"can\x89Ûªt", "cannot", tweet)
    tweet = re.sub(r"would've", "would have", tweet)
    tweet = re.sub(r"it'll", "it will", tweet)
    tweet = re.sub(r"we'll", "we will", tweet)
    tweet = re.sub(r"wouldn\x89Ûªt", "would not", tweet)
    tweet = re.sub(r"We've", "We have", tweet)
    tweet = re.sub(r"he'll", "he will", tweet)
    tweet = re.sub(r"Y'all", "You all", tweet)
    tweet = re.sub(r"Weren't", "Were not", tweet)
    tweet = re.sub(r"Didn't", "Did not", tweet)
    tweet = re.sub(r"they'll", "they will", tweet)
    tweet = re.sub(r"they'd", "they would", tweet)
    tweet = re.sub(r"DON'T", "DO NOT", tweet)
    tweet = re.sub(r"That\x89Ûªs", "That is", tweet)
    tweet = re.sub(r"they've", "they have", tweet)
    tweet = re.sub(r"i'd", "I would", tweet)
    tweet = re.sub(r"should've", "should have", tweet)
    tweet = re.sub(r"You\x89Ûªre", "You are", tweet)
    tweet = re.sub(r"where's", "where is", tweet)
    tweet = re.sub(r"Don\x89Ûªt", "Do not", tweet)
    tweet = re.sub(r"we'd", "we would", tweet)
    tweet = re.sub(r"i'll", "I will", tweet)
    tweet = re.sub(r"weren't", "were not", tweet)
    tweet = re.sub(r"They're", "They are", tweet)
    tweet = re.sub(r"Can\x89Ûªt", "Cannot", tweet)
    tweet = re.sub(r"you\x89Ûªll", "you will", tweet)
    tweet = re.sub(r"I\x89Ûªd", "I would", tweet)
    tweet = re.sub(r"let's", "let us", tweet)
    tweet = re.sub(r"it's", "it is", tweet)
    tweet = re.sub(r"can't", "cannot", tweet)
    tweet = re.sub(r"don't", "do not", tweet)
    tweet = re.sub(r"you're", "you are", tweet)
    tweet = re.sub(r"i've", "I have", tweet)
    tweet = re.sub(r"that's", "that is", tweet)
    tweet = re.sub(r"i'll", "I will", tweet)
    tweet = re.sub(r"doesn't", "does not", tweet)
    tweet = re.sub(r"i'd", "I would", tweet)
    tweet = re.sub(r"didn't", "did not", tweet)
    tweet = re.sub(r"ain't", "am not", tweet)
    tweet = re.sub(r"you'll", "you will", tweet)
    tweet = re.sub(r"I've", "I have", tweet)
    tweet = re.sub(r"Don't", "do not", tweet)
    tweet = re.sub(r"I'll", "I will", tweet)
    tweet = re.sub(r"I'd", "I would", tweet)
    tweet = re.sub(r"Let's", "Let us", tweet)
    tweet = re.sub(r"you'd", "You would", tweet)
    tweet = re.sub(r"It's", "It is", tweet)
    tweet = re.sub(r"Ain't", "am not", tweet)
    tweet = re.sub(r"Haven't", "Have not", tweet)
    tweet = re.sub(r"Could've", "Could have", tweet)
    tweet = re.sub(r"youve", "you have", tweet)
    tweet = re.sub(r"donå«t", "do not", tweet)

    # ── CHARACTER ENTITY REFERENCES ─────────────────────────────────────────
    tweet = re.sub(r"&gt;", ">", tweet)
    tweet = re.sub(r"&lt;", "<", tweet)
    tweet = re.sub(r"&amp;", "&", tweet)

    # ── TYPOS, SLANG & ABBREVIATIONS ────────────────────────────────────────
    tweet = re.sub(r"w/e", "whatever", tweet)
    tweet = re.sub(r"w/", "with", tweet)
    tweet = re.sub(r"USAgov", "USA government", tweet)
    tweet = re.sub(r"recentlu", "recently", tweet)
    tweet = re.sub(r"Ph0tos", "Photos", tweet)
    tweet = re.sub(r"amirite", "am I right", tweet)
    tweet = re.sub(r"exp0sed", "exposed", tweet)
    tweet = re.sub(r"<3", "love", tweet)
    tweet = re.sub(r"amageddon", "armageddon", tweet)
    tweet = re.sub(r"Trfc", "Traffic", tweet)
    tweet = re.sub(r"8/5/2015", "2015-08-05", tweet)
    tweet = re.sub(r"WindStorm", "Wind Storm", tweet)
    tweet = re.sub(r"8/6/2015", "2015-08-06", tweet)
    tweet = re.sub(r"10:38PM", "10:38 PM", tweet)
    tweet = re.sub(r"10:30pm", "10:30 PM", tweet)
    tweet = re.sub(r"16yr", "16 year", tweet)
    tweet = re.sub(r"lmao", "laughing my ass off", tweet)
    tweet = re.sub(r"TRAUMATISED", "traumatized", tweet)

    # ── HASHTAGS & USERNAMES CAS PAR CAS ────────────────────────────────────
    tweet = re.sub(r"IranDeal", "Iran Deal", tweet)
    tweet = re.sub(r"ArianaGrande", "Ariana Grande", tweet)
    tweet = re.sub(r"camilacabello97", "camila cabello", tweet)
    tweet = re.sub(r"RondaRousey", "Ronda Rousey", tweet)
    tweet = re.sub(r"MTVHottest", "MTV Hottest", tweet)
    tweet = re.sub(r"TrapMusic", "Trap Music", tweet)
    tweet = re.sub(r"ProphetMuhammad", "Prophet Muhammad", tweet)
    tweet = re.sub(r"PantherAttack", "Panther Attack", tweet)
    tweet = re.sub(r"StrategicPatience", "Strategic Patience", tweet)
    tweet = re.sub(r"socialnews", "social news", tweet)
    tweet = re.sub(r"NASAHurricane", "NASA Hurricane", tweet)
    tweet = re.sub(r"onlinecommunities", "online communities", tweet)
    tweet = re.sub(r"humanconsumption", "human consumption", tweet)
    tweet = re.sub(r"Typhoon-Devastated", "Typhoon Devastated", tweet)
    tweet = re.sub(r"Meat-Loving", "Meat Loving", tweet)
    tweet = re.sub(r"facialabuse", "facial abuse", tweet)
    tweet = re.sub(r"LakeCounty", "Lake County", tweet)
    tweet = re.sub(r"BeingAuthor", "Being Author", tweet)
    tweet = re.sub(r"withheavenly", "with heavenly", tweet)
    tweet = re.sub(r"thankU", "thank you", tweet)
    tweet = re.sub(r"iTunesMusic", "iTunes Music", tweet)
    tweet = re.sub(r"OffensiveContent", "Offensive Content", tweet)
    tweet = re.sub(r"WorstSummerJob", "Worst Summer Job", tweet)
    tweet = re.sub(r"HarryBeCareful", "Harry Be Careful", tweet)
    tweet = re.sub(r"NASASolarSystem", "NASA Solar System", tweet)
    tweet = re.sub(r"animalrescue", "animal rescue", tweet)
    tweet = re.sub(r"KurtSchlichter", "Kurt Schlichter", tweet)
    tweet = re.sub(r"aRmageddon", "armageddon", tweet)
    tweet = re.sub(r"Throwingknifes", "Throwing knives", tweet)
    tweet = re.sub(r"GodsLove", "God's Love", tweet)
    tweet = re.sub(r"bookboost", "book boost", tweet)
    tweet = re.sub(r"ibooklove", "I book love", tweet)
    tweet = re.sub(r"NestleIndia", "Nestle India", tweet)
    tweet = re.sub(r"realDonaldTrump", "Donald Trump", tweet)
    tweet = re.sub(r"DavidVonderhaar", "David Vonderhaar", tweet)
    tweet = re.sub(r"CecilTheLion", "Cecil The Lion", tweet)
    tweet = re.sub(r"weathernetwork", "weather network", tweet)
    tweet = re.sub(r"withBioterrorism&use", "with Bioterrorism & use", tweet)
    tweet = re.sub(r"Hostage&2", "Hostage & 2", tweet)
    tweet = re.sub(r"GOPDebate", "GOP Debate", tweet)
    tweet = re.sub(r"RickPerry", "Rick Perry", tweet)
    tweet = re.sub(r"frontpage", "front page", tweet)
    tweet = re.sub(r"NewsInTweets", "News In Tweets", tweet)
    tweet = re.sub(r"ViralSpell", "Viral Spell", tweet)
    tweet = re.sub(r"til_now", "until now", tweet)
    tweet = re.sub(r"volcanoinRussia", "volcano in Russia", tweet)
    tweet = re.sub(r"ZippedNews", "Zipped News", tweet)
    tweet = re.sub(r"MicheleBachman", "Michele Bachman", tweet)
    tweet = re.sub(r"53inch", "53 inch", tweet)
    tweet = re.sub(r"KerrickTrial", "Kerrick Trial", tweet)
    tweet = re.sub(r"abstorm", "Alberta Storm", tweet)
    tweet = re.sub(r"Beyhive", "Beyonce hive", tweet)
    tweet = re.sub(r"IDFire", "Idaho Fire", tweet)
    tweet = re.sub(r"DETECTADO", "Detected", tweet)
    tweet = re.sub(r"RockyFire", "Rocky Fire", tweet)
    tweet = re.sub(r"Listen/Buy", "Listen / Buy", tweet)
    tweet = re.sub(r"NickCannon", "Nick Cannon", tweet)
    tweet = re.sub(r"FaroeIslands", "Faroe Islands", tweet)
    tweet = re.sub(r"yycstorm", "Calgary Storm", tweet)
    tweet = re.sub(r"IDPs:", "Internally Displaced People :", tweet)
    tweet = re.sub(r"ArtistsUnited", "Artists United", tweet)
    tweet = re.sub(r"ClaytonBryant", "Clayton Bryant", tweet)
    tweet = re.sub(r"jimmyfallon", "jimmy fallon", tweet)
    tweet = re.sub(r"justinbieber", "justin bieber", tweet)
    tweet = re.sub(r"UTC2015", "UTC 2015", tweet)
    tweet = re.sub(r"Time2015", "Time 2015", tweet)
    tweet = re.sub(r"djicemoon", "dj icemoon", tweet)
    tweet = re.sub(r"LivingSafely", "Living Safely", tweet)
    tweet = re.sub(r"FIFA16", "Fifa 2016", tweet)
    tweet = re.sub(r"thisiswhywecanthavenicethings", "this is why we cannot have nice things", tweet)
    tweet = re.sub(r"bbcnews", "bbc news", tweet)
    tweet = re.sub(r"UndergroundRailraod", "Underground Railraod", tweet)
    tweet = re.sub(r"c4news", "c4 news", tweet)
    tweet = re.sub(r"OBLITERATION", "obliteration", tweet)
    tweet = re.sub(r"MUDSLIDE", "mudslide", tweet)
    tweet = re.sub(r"NoSurrender", "No Surrender", tweet)
    tweet = re.sub(r"NotExplained", "Not Explained", tweet)
    tweet = re.sub(r"greatbritishbakeoff", "great british bake off", tweet)
    tweet = re.sub(r"LondonFire", "London Fire", tweet)
    tweet = re.sub(r"KOTAWeather", "KOTA Weather", tweet)
    tweet = re.sub(r"LuchaUnderground", "Lucha Underground", tweet)
    tweet = re.sub(r"KOIN6News", "KOIN 6 News", tweet)
    tweet = re.sub(r"LiveOnK2", "Live On K2", tweet)
    tweet = re.sub(r"9NewsGoldCoast", "9 News Gold Coast", tweet)
    tweet = re.sub(r"nikeplus", "nike plus", tweet)
    tweet = re.sub(r"david_cameron", "David Cameron", tweet)
    tweet = re.sub(r"peterjukes", "Peter Jukes", tweet)
    tweet = re.sub(r"JamesMelville", "James Melville", tweet)
    tweet = re.sub(r"megynkelly", "Megyn Kelly", tweet)
    tweet = re.sub(r"cnewslive", "C News Live", tweet)
    tweet = re.sub(r"JamaicaObserver", "Jamaica Observer", tweet)
    tweet = re.sub(r"TweetLikeItsSeptember11th2001", "Tweet like it is september 11th 2001", tweet)
    tweet = re.sub(r"cbplawyers", "cbp lawyers", tweet)
    tweet = re.sub(r"fewmoretweets", "few more tweets", tweet)
    tweet = re.sub(r"BlackLivesMatter", "Black Lives Matter", tweet)
    tweet = re.sub(r"cjoyner", "Chris Joyner", tweet)
    tweet = re.sub(r"ENGvAUS", "England vs Australia", tweet)
    tweet = re.sub(r"ScottWalker", "Scott Walker", tweet)
    tweet = re.sub(r"MikeParrActor", "Michael Parr", tweet)
    tweet = re.sub(r"4PlayThursdays", "Foreplay Thursdays", tweet)
    tweet = re.sub(r"TGF2015", "Tontitown Grape Festival", tweet)
    tweet = re.sub(r"realmandyrain", "Mandy Rain", tweet)
    tweet = re.sub(r"GraysonDolan", "Grayson Dolan", tweet)
    tweet = re.sub(r"ApolloBrown", "Apollo Brown", tweet)
    tweet = re.sub(r"saddlebrooke", "Saddlebrooke", tweet)
    tweet = re.sub(r"TontitownGrape", "Tontitown Grape", tweet)
    tweet = re.sub(r"AbbsWinston", "Abbs Winston", tweet)
    tweet = re.sub(r"ShaunKing", "Shaun King", tweet)
    tweet = re.sub(r"MeekMill", "Meek Mill", tweet)
    tweet = re.sub(r"TornadoGiveaway", "Tornado Giveaway", tweet)
    tweet = re.sub(r"GRupdates", "GR updates", tweet)
    tweet = re.sub(r"SouthDowns", "South Downs", tweet)
    tweet = re.sub(r"braininjury", "brain injury", tweet)
    tweet = re.sub(r"auspol", "Australian politics", tweet)
    tweet = re.sub(r"PlannedParenthood", "Planned Parenthood", tweet)
    tweet = re.sub(r"calgaryweather", "Calgary Weather", tweet)
    tweet = re.sub(r"weallheartonedirection", "we all heart one direction", tweet)
    tweet = re.sub(r"edsheeran", "Ed Sheeran", tweet)
    tweet = re.sub(r"TrueHeroes", "True Heroes", tweet)
    tweet = re.sub(r"S3XLEAK", "sex leak", tweet)
    tweet = re.sub(r"ComplexMag", "Complex Magazine", tweet)
    tweet = re.sub(r"TheAdvocateMag", "The Advocate Magazine", tweet)
    tweet = re.sub(r"CityofCalgary", "City of Calgary", tweet)
    tweet = re.sub(r"EbolaOutbreak", "Ebola Outbreak", tweet)
    tweet = re.sub(r"SummerFate", "Summer Fate", tweet)
    tweet = re.sub(r"RAmag", "Royal Academy Magazine", tweet)
    tweet = re.sub(r"offers2go", "offers to go", tweet)
    tweet = re.sub(r"foodscare", "food scare", tweet)
    tweet = re.sub(r"MNPDNashville", "Metropolitan Nashville Police Department", tweet)
    tweet = re.sub(r"TfLBusAlerts", "TfL Bus Alerts", tweet)
    tweet = re.sub(r"GamerGate", "Gamer Gate", tweet)
    tweet = re.sub(r"IHHen", "Humanitarian Relief", tweet)
    tweet = re.sub(r"spinningbot", "spinning bot", tweet)
    tweet = re.sub(r"ModiMinistry", "Modi Ministry", tweet)
    tweet = re.sub(r"TAXIWAYS", "taxi ways", tweet)
    tweet = re.sub(r"Calum5SOS", "Calum Hood", tweet)
    tweet = re.sub(r"po_st", "po.st", tweet)
    tweet = re.sub(r"scoopit", "scoop.it", tweet)
    tweet = re.sub(r"UltimaLucha", "Ultima Lucha", tweet)
    tweet = re.sub(r"JonathanFerrell", "Jonathan Ferrell", tweet)
    tweet = re.sub(r"aria_ahrary", "Aria Ahrary", tweet)
    tweet = re.sub(r"rapidcity", "Rapid City", tweet)
    tweet = re.sub(r"OutBid", "outbid", tweet)
    tweet = re.sub(r"lavenderpoetrycafe", "lavender poetry cafe", tweet)
    tweet = re.sub(r"EudryLantiqua", "Eudry Lantiqua", tweet)
    tweet = re.sub(r"15PM", "15 PM", tweet)
    tweet = re.sub(r"OriginalFunko", "Funko", tweet)
    tweet = re.sub(r"rightwaystan", "Richard Tan", tweet)
    tweet = re.sub(r"CindyNoonan", "Cindy Noonan", tweet)
    tweet = re.sub(r"RT_America", "RT America", tweet)
    tweet = re.sub(r"narendramodi", "Narendra Modi", tweet)
    tweet = re.sub(r"BakeOffFriends", "Bake Off Friends", tweet)
    tweet = re.sub(r"TeamHendrick", "Hendrick Motorsports", tweet)
    tweet = re.sub(r"alexbelloli", "Alex Belloli", tweet)
    tweet = re.sub(r"itsjustinstuart", "Justin Stuart", tweet)
    tweet = re.sub(r"gunsense", "gun sense", tweet)
    tweet = re.sub(r"DebateQuestionsWeWantToHear", "debate questions we want to hear", tweet)
    tweet = re.sub(r"RoyalCarribean", "Royal Carribean", tweet)
    tweet = re.sub(r"samanthaturne19", "Samantha Turner", tweet)
    tweet = re.sub(r"JonVoyage", "Jon Stewart", tweet)
    tweet = re.sub(r"renew911health", "renew 911 health", tweet)
    tweet = re.sub(r"SuryaRay", "Surya Ray", tweet)
    tweet = re.sub(r"pattonoswalt", "Patton Oswalt", tweet)
    tweet = re.sub(r"minhazmerchant", "Minhaz Merchant", tweet)
    tweet = re.sub(r"TLVFaces", "Israel Diaspora Coalition", tweet)
    tweet = re.sub(r"pmarca", "Marc Andreessen", tweet)
    tweet = re.sub(r"pdx911", "Portland Police", tweet)
    tweet = re.sub(r"jamaicaplain", "Jamaica Plain", tweet)
    tweet = re.sub(r"Japton", "Arkansas", tweet)
    tweet = re.sub(r"RouteComplex", "Route Complex", tweet)
    tweet = re.sub(r"INSubcontinent", "Indian Subcontinent", tweet)
    tweet = re.sub(r"NJTurnpike", "New Jersey Turnpike", tweet)
    tweet = re.sub(r"Politifiact", "PolitiFact", tweet)
    tweet = re.sub(r"Hiroshima70", "Hiroshima", tweet)
    tweet = re.sub(r"GMMBC", "Greater Mt Moriah Baptist Church", tweet)
    tweet = re.sub(r"versethe", "verse the", tweet)
    tweet = re.sub(r"TubeStrike", "Tube Strike", tweet)
    tweet = re.sub(r"MissionHills", "Mission Hills", tweet)
    tweet = re.sub(r"ProtectDenaliWolves", "Protect Denali Wolves", tweet)
    tweet = re.sub(r"NANKANA", "Nankana", tweet)
    tweet = re.sub(r"SAHIB", "Sahib", tweet)
    tweet = re.sub(r"PAKPATTAN", "Pakpattan", tweet)
    tweet = re.sub(r"Newz_Sacramento", "News Sacramento", tweet)
    tweet = re.sub(r"gofundme", "go fund me", tweet)
    tweet = re.sub(r"pmharper", "Stephen Harper", tweet)
    tweet = re.sub(r"IvanBerroa", "Ivan Berroa", tweet)
    tweet = re.sub(r"LosDelSonido", "Los Del Sonido", tweet)
    tweet = re.sub(r"bancodeseries", "banco de series", tweet)
    tweet = re.sub(r"timkaine", "Tim Kaine", tweet)
    tweet = re.sub(r"IdentityTheft", "Identity Theft", tweet)
    tweet = re.sub(r"AllLivesMatter", "All Lives Matter", tweet)
    tweet = re.sub(r"mishacollins", "Misha Collins", tweet)
    tweet = re.sub(r"BillNeelyNBC", "Bill Neely", tweet)
    tweet = re.sub(r"BeClearOnCancer", "be clear on cancer", tweet)
    tweet = re.sub(r"Kowing", "Knowing", tweet)
    tweet = re.sub(r"ScreamQueens", "Scream Queens", tweet)
    tweet = re.sub(r"AskCharley", "Ask Charley", tweet)
    tweet = re.sub(r"BlizzHeroes", "Heroes of the Storm", tweet)
    tweet = re.sub(r"BradleyBrad47", "Bradley Brad", tweet)
    tweet = re.sub(r"HannaPH", "Typhoon Hanna", tweet)
    tweet = re.sub(r"meinlcymbals", "MEINL Cymbals", tweet)
    tweet = re.sub(r"Ptbo", "Peterborough", tweet)
    tweet = re.sub(r"cnnbrk", "CNN Breaking News", tweet)
    tweet = re.sub(r"IndianNews", "Indian News", tweet)
    tweet = re.sub(r"savebees", "save bees", tweet)
    tweet = re.sub(r"GreenHarvard", "Green Harvard", tweet)
    tweet = re.sub(r"StandwithPP", "Stand with planned parenthood", tweet)
    tweet = re.sub(r"hermancranston", "Herman Cranston", tweet)
    tweet = re.sub(r"WMUR9", "WMUR-TV", tweet)
    tweet = re.sub(r"RockBottomRadFM", "Rock Bottom Radio", tweet)
    tweet = re.sub(r"ameenshaikh3", "Ameen Shaikh", tweet)
    tweet = re.sub(r"ProSyn", "Project Syndicate", tweet)
    tweet = re.sub(r"Daesh", "ISIS", tweet)
    tweet = re.sub(r"s2g", "swear to god", tweet)
    tweet = re.sub(r"listenlive", "listen live", tweet)
    tweet = re.sub(r"CDCgov", "Centers for Disease Control and Prevention", tweet)
    tweet = re.sub(r"FoxNew", "Fox News", tweet)
    tweet = re.sub(r"CBSBigBrother", "Big Brother", tweet)
    tweet = re.sub(r"JulieDiCaro", "Julie DiCaro", tweet)
    tweet = re.sub(r"theadvocatemag", "The Advocate Magazine", tweet)
    tweet = re.sub(r"RohnertParkDPS", "Rohnert Park Police Department", tweet)
    tweet = re.sub(r"THISIZBWRIGHT", "Bonnie Wright", tweet)
    tweet = re.sub(r"Popularmmos", "Popular MMOs", tweet)
    tweet = re.sub(r"WildHorses", "Wild Horses", tweet)
    tweet = re.sub(r"FantasticFour", "Fantastic Four", tweet)
    tweet = re.sub(r"HORNDALE", "Horndale", tweet)
    tweet = re.sub(r"PINER", "Piner", tweet)
    tweet = re.sub(r"BathAndNorthEastSomerset", "Bath and North East Somerset", tweet)
    tweet = re.sub(r"thatswhatfriendsarefor", "that is what friends are for", tweet)
    tweet = re.sub(r"residualincome", "residual income", tweet)
    tweet = re.sub(r"YahooNewsDigest", "Yahoo News Digest", tweet)
    tweet = re.sub(r"MalaysiaAirlines", "Malaysia Airlines", tweet)
    tweet = re.sub(r"AmazonDeals", "Amazon Deals", tweet)
    tweet = re.sub(r"MissCharleyWebb", "Charley Webb", tweet)
    tweet = re.sub(r"shoalstraffic", "shoals traffic", tweet)
    tweet = re.sub(r"GeorgeFoster72", "George Foster", tweet)
    tweet = re.sub(r"pop2015", "pop 2015", tweet)
    tweet = re.sub(r"_PokemonCards_", "Pokemon Cards", tweet)
    tweet = re.sub(r"DianneG", "Dianne Gallagher", tweet)
    tweet = re.sub(r"KashmirConflict", "Kashmir Conflict", tweet)
    tweet = re.sub(r"BritishBakeOff", "British Bake Off", tweet)
    tweet = re.sub(r"FreeKashmir", "Free Kashmir", tweet)
    tweet = re.sub(r"mattmosley", "Matt Mosley", tweet)
    tweet = re.sub(r"BishopFred", "Bishop Fred", tweet)
    tweet = re.sub(r"EndConflict", "End Conflict", tweet)
    tweet = re.sub(r"EndOccupation", "End Occupation", tweet)
    tweet = re.sub(r"UNHEALED", "unhealed", tweet)
    tweet = re.sub(r"CharlesDagnall", "Charles Dagnall", tweet)
    tweet = re.sub(r"Latestnews", "Latest news", tweet)
    tweet = re.sub(r"KindleCountdown", "Kindle Countdown", tweet)
    tweet = re.sub(r"NoMoreHandouts", "No More Handouts", tweet)
    tweet = re.sub(r"datingtips", "dating tips", tweet)
    tweet = re.sub(r"charlesadler", "Charles Adler", tweet)
    tweet = re.sub(r"twia", "Texas Windstorm Insurance Association", tweet)
    tweet = re.sub(r"txlege", "Texas Legislature", tweet)
    tweet = re.sub(r"WindstormInsurer", "Windstorm Insurer", tweet)
    tweet = re.sub(r"Newss", "News", tweet)
    tweet = re.sub(r"hempoil", "hemp oil", tweet)
    tweet = re.sub(r"CommoditiesAre", "Commodities are", tweet)
    tweet = re.sub(r"tubestrike", "tube strike", tweet)
    tweet = re.sub(r"JoeNBC", "Joe Scarborough", tweet)
    tweet = re.sub(r"LiteraryCakes", "Literary Cakes", tweet)
    tweet = re.sub(r"TI5", "The International 5", tweet)
    tweet = re.sub(r"thehill", "the hill", tweet)
    tweet = re.sub(r"3others", "3 others", tweet)
    tweet = re.sub(r"stighefootball", "Sam Tighe", tweet)
    tweet = re.sub(r"whatstheimportantvideo", "what is the important video", tweet)
    tweet = re.sub(r"ClaudioMeloni", "Claudio Meloni", tweet)
    tweet = re.sub(r"DukeSkywalker", "Duke Skywalker", tweet)
    tweet = re.sub(r"carsonmwr", "Fort Carson", tweet)
    tweet = re.sub(r"offdishduty", "off dish duty", tweet)
    tweet = re.sub(r"andword", "and word", tweet)
    tweet = re.sub(r"rhodeisland", "Rhode Island", tweet)
    tweet = re.sub(r"easternoregon", "Eastern Oregon", tweet)
    tweet = re.sub(r"WAwildfire", "Washington Wildfire", tweet)
    tweet = re.sub(r"fingerrockfire", "Finger Rock Fire", tweet)
    tweet = re.sub(r"57am", "57 am", tweet)
    tweet = re.sub(r"JacobHoggard", "Jacob Hoggard", tweet)
    tweet = re.sub(r"newnewnew", "new new new", tweet)
    tweet = re.sub(r"under50", "under 50", tweet)
    tweet = re.sub(r"getitbeforeitsgone", "get it before it is gone", tweet)
    tweet = re.sub(r"freshoutofthebox", "fresh out of the box", tweet)
    tweet = re.sub(r"amwriting", "am writing", tweet)
    tweet = re.sub(r"Bokoharm", "Boko Haram", tweet)
    tweet = re.sub(r"Nowlike", "Now like", tweet)
    tweet = re.sub(r"seasonfrom", "season from", tweet)
    tweet = re.sub(r"epicente", "epicenter", tweet)
    tweet = re.sub(r"epicenterr", "epicenter", tweet)
    tweet = re.sub(r"sicklife", "sick life", tweet)
    tweet = re.sub(r"yycweather", "Calgary Weather", tweet)
    tweet = re.sub(r"calgarysun", "Calgary Sun", tweet)
    tweet = re.sub(r"approachng", "approaching", tweet)
    tweet = re.sub(r"evng", "evening", tweet)
    tweet = re.sub(r"Sumthng", "something", tweet)
    tweet = re.sub(r"EllenPompeo", "Ellen Pompeo", tweet)
    tweet = re.sub(r"shondarhimes", "Shonda Rhimes", tweet)
    tweet = re.sub(r"ABCNetwork", "ABC Network", tweet)
    tweet = re.sub(r"SushmaSwaraj", "Sushma Swaraj", tweet)
    tweet = re.sub(r"pray4japan", "Pray for Japan", tweet)
    tweet = re.sub(r"hope4japan", "Hope for Japan", tweet)
    tweet = re.sub(r"Illusionimagess", "Illusion images", tweet)
    tweet = re.sub(r"SummerUnderTheStars", "Summer Under The Stars", tweet)
    tweet = re.sub(r"ShallWeDance", "Shall We Dance", tweet)
    tweet = re.sub(r"TCMParty", "TCM Party", tweet)
    tweet = re.sub(r"marijuananews", "marijuana news", tweet)
    tweet = re.sub(r"onbeingwithKristaTippett", "on being with Krista Tippett", tweet)
    tweet = re.sub(r"Beingtweets", "Being tweets", tweet)
    tweet = re.sub(r"newauthors", "new authors", tweet)
    tweet = re.sub(r"remedyyyy", "remedy", tweet)
    tweet = re.sub(r"44PM", "44 PM", tweet)
    tweet = re.sub(r"HeadlinesApp", "Headlines App", tweet)
    tweet = re.sub(r"40PM", "40 PM", tweet)
    tweet = re.sub(r"myswc", "Severe Weather Center", tweet)
    tweet = re.sub(r"ithats", "that is", tweet)
    tweet = re.sub(r"icouldsitinthismomentforever", "I could sit in this moment forever", tweet)
    tweet = re.sub(r"FatLoss", "Fat Loss", tweet)
    tweet = re.sub(r"02PM", "02 PM", tweet)
    tweet = re.sub(r"MetroFmTalk", "Metro Fm Talk", tweet)
    tweet = re.sub(r"Bstrd", "bastard", tweet)
    tweet = re.sub(r"bldy", "bloody", tweet)
    tweet = re.sub(r"MetrofmTalk", "Metro Fm Talk", tweet)
    tweet = re.sub(r"terrorismturn", "terrorism turn", tweet)
    tweet = re.sub(r"BBCNewsAsia", "BBC News Asia", tweet)
    tweet = re.sub(r"BehindTheScenes", "Behind The Scenes", tweet)
    tweet = re.sub(r"GeorgeTakei", "George Takei", tweet)
    tweet = re.sub(r"WomensWeeklyMag", "Womens Weekly Magazine", tweet)
    tweet = re.sub(r"SurvivorsGuidetoEarth", "Survivors Guide to Earth", tweet)
    tweet = re.sub(r"incubusband", "incubus band", tweet)
    tweet = re.sub(r"Babypicturethis", "Baby picture this", tweet)
    tweet = re.sub(r"BombEffects", "Bomb Effects", tweet)
    tweet = re.sub(r"win10", "Windows 10", tweet)
    tweet = re.sub(r"idkidk", "I do not know I do not know", tweet)
    tweet = re.sub(r"TheWalkingDead", "The Walking Dead", tweet)
    tweet = re.sub(r"amyschumer", "Amy Schumer", tweet)
    tweet = re.sub(r"crewlist", "crew list", tweet)
    tweet = re.sub(r"Erdogans", "Erdogan", tweet)
    tweet = re.sub(r"BBCLive", "BBC Live", tweet)
    tweet = re.sub(r"TonyAbbottMHR", "Tony Abbott", tweet)
    tweet = re.sub(r"paulmyerscough", "Paul Myerscough", tweet)
    tweet = re.sub(r"georgegallagher", "George Gallagher", tweet)
    tweet = re.sub(r"JimmieJohnson", "Jimmie Johnson", tweet)
    tweet = re.sub(r"pctool", "pc tool", tweet)
    tweet = re.sub(r"DoingHashtagsRight", "Doing Hashtags Right", tweet)
    tweet = re.sub(r"ThrowbackThursday", "Throwback Thursday", tweet)
    tweet = re.sub(r"SnowBackSunday", "Snowback Sunday", tweet)
    tweet = re.sub(r"LakeEffect", "Lake Effect", tweet)
    tweet = re.sub(r"RTphotographyUK", "Richard Thomas Photography UK", tweet)
    tweet = re.sub(r"BigBang_CBS", "Big Bang CBS", tweet)
    tweet = re.sub(r"writerslife", "writers life", tweet)
    tweet = re.sub(r"NaturalBirth", "Natural Birth", tweet)
    tweet = re.sub(r"UnusualWords", "Unusual Words", tweet)
    tweet = re.sub(r"wizkhalifa", "Wiz Khalifa", tweet)
    tweet = re.sub(r"acreativedc", "a creative DC", tweet)
    tweet = re.sub(r"vscodc", "vsco DC", tweet)
    tweet = re.sub(r"VSCOcam", "vsco camera", tweet)
    tweet = re.sub(r"TheBEACHDC", "The beach DC", tweet)
    tweet = re.sub(r"buildingmuseum", "building museum", tweet)
    tweet = re.sub(r"WorldOil", "World Oil", tweet)
    tweet = re.sub(r"redwedding", "red wedding", tweet)
    tweet = re.sub(r"AmazingRaceCanada", "Amazing Race Canada", tweet)
    tweet = re.sub(r"WakeUpAmerica", "Wake Up America", tweet)
    tweet = re.sub(r"\\Allahuakbar\\", "Allahu Akbar", tweet)
    tweet = re.sub(r"bleased", "blessed", tweet)
    tweet = re.sub(r"nigeriantribune", "Nigerian Tribune", tweet)
    tweet = re.sub(r"HIDEO_KOJIMA_EN", "Hideo Kojima", tweet)
    tweet = re.sub(r"FusionFestival", "Fusion Festival", tweet)
    tweet = re.sub(r"50Mixed", "50 Mixed", tweet)
    tweet = re.sub(r"NoAgenda", "No Agenda", tweet)
    tweet = re.sub(r"WhiteGenocide", "White Genocide", tweet)
    tweet = re.sub(r"dirtylying", "dirty lying", tweet)
    tweet = re.sub(r"SyrianRefugees", "Syrian Refugees", tweet)
    tweet = re.sub(r"changetheworld", "change the world", tweet)
    tweet = re.sub(r"Ebolacase", "Ebola case", tweet)
    tweet = re.sub(r"mcgtech", "mcg technologies", tweet)
    tweet = re.sub(r"withweapons", "with weapons", tweet)
    tweet = re.sub(r"advancedwarfare", "advanced warfare", tweet)
    tweet = re.sub(r"letsFootball", "let us Football", tweet)
    tweet = re.sub(r"LateNiteMix", "late night mix", tweet)
    tweet = re.sub(r"PhilCollinsFeed", "Phil Collins", tweet)
    tweet = re.sub(r"RudyHavenstein", "Rudy Havenstein", tweet)
    tweet = re.sub(r"22PM", "22 PM", tweet)
    tweet = re.sub(r"54am", "54 AM", tweet)
    tweet = re.sub(r"38am", "38 AM", tweet)
    tweet = re.sub(r"OldFolkExplainStuff", "Old Folk Explain Stuff", tweet)
    tweet = re.sub(r"BlacklivesMatter", "Black Lives Matter", tweet)
    tweet = re.sub(r"InsaneLimits", "Insane Limits", tweet)
    tweet = re.sub(r"youcantsitwithus", "you cannot sit with us", tweet)
    tweet = re.sub(r"2k15", "2015", tweet)
    tweet = re.sub(r"TheIran", "Iran", tweet)
    tweet = re.sub(r"JimmyFallon", "Jimmy Fallon", tweet)
    tweet = re.sub(r"AlbertBrooks", "Albert Brooks", tweet)
    tweet = re.sub(r"defense_news", "defense news", tweet)
    tweet = re.sub(r"nuclearrcSA", "Nuclear Risk Control Self Assessment", tweet)
    tweet = re.sub(r"Auspol", "Australia Politics", tweet)
    tweet = re.sub(r"NuclearPower", "Nuclear Power", tweet)
    tweet = re.sub(r"WhiteTerrorism", "White Terrorism", tweet)
    tweet = re.sub(r"truthfrequencyradio", "Truth Frequency Radio", tweet)
    tweet = re.sub(r"ErasureIsNotEquality", "Erasure is not equality", tweet)
    tweet = re.sub(r"ProBonoNews", "Pro Bono News", tweet)
    tweet = re.sub(r"JakartaPost", "Jakarta Post", tweet)
    tweet = re.sub(r"toopainful", "too painful", tweet)
    tweet = re.sub(r"melindahaunton", "Melinda Haunton", tweet)
    tweet = re.sub(r"NoNukes", "No Nukes", tweet)
    tweet = re.sub(r"curryspcworld", "Currys PC World", tweet)
    tweet = re.sub(r"ineedcake", "I need cake", tweet)
    tweet = re.sub(r"blackforestgateau", "black forest gateau", tweet)
    tweet = re.sub(r"BBCOne", "BBC One", tweet)
    tweet = re.sub(r"AlexxPage", "Alex Page", tweet)
    tweet = re.sub(r"jonathanserrie", "Jonathan Serrie", tweet)
    tweet = re.sub(r"SocialJerkBlog", "Social Jerk Blog", tweet)
    tweet = re.sub(r"ChelseaVPeretti", "Chelsea Peretti", tweet)
    tweet = re.sub(r"irongiant", "iron giant", tweet)
    tweet = re.sub(r"RonFunches", "Ron Funches", tweet)
    tweet = re.sub(r"TimCook", "Tim Cook", tweet)
    tweet = re.sub(r"sebastianstanisaliveandwell", "Sebastian Stan is alive and well", tweet)
    tweet = re.sub(r"Madsummer", "Mad summer", tweet)
    tweet = re.sub(r"NowYouKnow", "Now you know", tweet)
    tweet = re.sub(r"concertphotography", "concert photography", tweet)
    tweet = re.sub(r"TomLandry", "Tom Landry", tweet)
    tweet = re.sub(r"showgirldayoff", "show girl day off", tweet)
    tweet = re.sub(r"Yougslavia", "Yugoslavia", tweet)
    tweet = re.sub(r"QuantumDataInformatics", "Quantum Data Informatics", tweet)
    tweet = re.sub(r"FromTheDesk", "From The Desk", tweet)
    tweet = re.sub(r"TheaterTrial", "Theater Trial", tweet)
    tweet = re.sub(r"CatoInstitute", "Cato Institute", tweet)
    tweet = re.sub(r"EmekaGift", "Emeka Gift", tweet)
    tweet = re.sub(r"LetsBe_Rational", "Let us be rational", tweet)
    tweet = re.sub(r"Cynicalreality", "Cynical reality", tweet)
    tweet = re.sub(r"FredOlsenCruise", "Fred Olsen Cruise", tweet)
    tweet = re.sub(r"NotSorry", "not sorry", tweet)
    tweet = re.sub(r"UseYourWords", "use your words", tweet)
    tweet = re.sub(r"WordoftheDay", "word of the day", tweet)
    tweet = re.sub(r"Dictionarycom", "Dictionary.com", tweet)
    tweet = re.sub(r"TheBrooklynLife", "The Brooklyn Life", tweet)
    tweet = re.sub(r"jokethey", "joke they", tweet)
    tweet = re.sub(r"nflweek1picks", "NFL week 1 picks", tweet)
    tweet = re.sub(r"uiseful", "useful", tweet)
    tweet = re.sub(r"JusticeDotOrg", "The American Association for Justice", tweet)
    tweet = re.sub(r"autoaccidents", "auto accidents", tweet)
    tweet = re.sub(r"SteveGursten", "Steve Gursten", tweet)
    tweet = re.sub(r"MichiganAutoLaw", "Michigan Auto Law", tweet)
    tweet = re.sub(r"birdgang", "bird gang", tweet)
    tweet = re.sub(r"nflnetwork", "NFL Network", tweet)
    tweet = re.sub(r"NYDNSports", "NY Daily News Sports", tweet)
    tweet = re.sub(r"RVacchianoNYDN", "Ralph Vacchiano NY Daily News", tweet)
    tweet = re.sub(r"EdmontonEsks", "Edmonton Eskimos", tweet)
    tweet = re.sub(r"david_brelsford", "David Brelsford", tweet)
    tweet = re.sub(r"TOI_India", "The Times of India", tweet)
    tweet = re.sub(r"hegot", "he got", tweet)
    tweet = re.sub(r"SkinsOn9", "Skins on 9", tweet)
    tweet = re.sub(r"sothathappened", "so that happened", tweet)
    tweet = re.sub(r"LCOutOfDoors", "LC Out Of Doors", tweet)
    tweet = re.sub(r"NationFirst", "Nation First", tweet)
    tweet = re.sub(r"IndiaToday", "India Today", tweet)
    tweet = re.sub(r"HLPS", "helps", tweet)
    tweet = re.sub(r"HOSTAGESTHROSW", "hostages throw", tweet)
    tweet = re.sub(r"SNCTIONS", "sanctions", tweet)
    tweet = re.sub(r"BidTime", "Bid Time", tweet)
    tweet = re.sub(r"crunchysensible", "crunchy sensible", tweet)
    tweet = re.sub(r"RandomActsOfRomance", "Random acts of romance", tweet)
    tweet = re.sub(r"MomentsAtHill", "Moments at hill", tweet)
    tweet = re.sub(r"eatshit", "eat shit", tweet)
    tweet = re.sub(r"liveleakfun", "live leak fun", tweet)
    tweet = re.sub(r"SahelNews", "Sahel News", tweet)
    tweet = re.sub(r"abc7newsbayarea", "ABC 7 News Bay Area", tweet)
    tweet = re.sub(r"facilitiesmanagement", "facilities management", tweet)
    tweet = re.sub(r"facilitydude", "facility dude", tweet)
    tweet = re.sub(r"CampLogistics", "Camp logistics", tweet)
    tweet = re.sub(r"alaskapublic", "Alaska public", tweet)
    tweet = re.sub(r"MarketResearch", "Market Research", tweet)
    tweet = re.sub(r"AccuracyEsports", "Accuracy Esports", tweet)
    tweet = re.sub(r"TheBodyShopAust", "The Body Shop Australia", tweet)
    tweet = re.sub(r"yychail", "Calgary hail", tweet)
    tweet = re.sub(r"yyctraffic", "Calgary traffic", tweet)
    tweet = re.sub(r"eliotschool", "eliot school", tweet)
    tweet = re.sub(r"TheBrokenCity", "The Broken City", tweet)
    tweet = re.sub(r"OldsFireDept", "Olds Fire Department", tweet)
    tweet = re.sub(r"RiverComplex", "River Complex", tweet)
    tweet = re.sub(r"fieldworksmells", "field work smells", tweet)
    tweet = re.sub(r"IranElection", "Iran Election", tweet)
    tweet = re.sub(r"glowng", "glowing", tweet)
    tweet = re.sub(r"kindlng", "kindling", tweet)
    tweet = re.sub(r"riggd", "rigged", tweet)
    tweet = re.sub(r"slownewsday", "slow news day", tweet)
    tweet = re.sub(r"MyanmarFlood", "Myanmar Flood", tweet)
    tweet = re.sub(r"abc7chicago", "ABC 7 Chicago", tweet)
    tweet = re.sub(r"copolitics", "Colorado Politics", tweet)
    tweet = re.sub(r"AdilGhumro", "Adil Ghumro", tweet)
    tweet = re.sub(r"netbots", "net bots", tweet)
    tweet = re.sub(r"byebyeroad", "bye bye road", tweet)
    tweet = re.sub(r"massiveflooding", "massive flooding", tweet)
    tweet = re.sub(r"EndofUS", "End of United States", tweet)
    tweet = re.sub(r"35PM", "35 PM", tweet)
    tweet = re.sub(r"greektheatrela", "Greek Theatre Los Angeles", tweet)
    tweet = re.sub(r"76mins", "76 minutes", tweet)
    tweet = re.sub(r"publicsafetyfirst", "public safety first", tweet)
    tweet = re.sub(r"livesmatter", "lives matter", tweet)
    tweet = re.sub(r"myhometown", "my hometown", tweet)
    tweet = re.sub(r"tankerfire", "tanker fire", tweet)
    tweet = re.sub(r"MEMORIALDAY", "memorial day", tweet)
    tweet = re.sub(r"MEMORIAL_DAY", "memorial day", tweet)
    tweet = re.sub(r"instaxbooty", "instagram booty", tweet)
    tweet = re.sub(r"Jerusalem_Post", "Jerusalem Post", tweet)
    tweet = re.sub(r"WayneRooney_INA", "Wayne Rooney", tweet)
    tweet = re.sub(r"VirtualReality", "Virtual Reality", tweet)
    tweet = re.sub(r"OculusRift", "Oculus Rift", tweet)
    tweet = re.sub(r"OwenJones84", "Owen Jones", tweet)
    tweet = re.sub(r"jeremycorbyn", "Jeremy Corbyn", tweet)
    tweet = re.sub(r"paulrogers002", "Paul Rogers", tweet)
    tweet = re.sub(r"mortalkombatx", "Mortal Kombat X", tweet)
    tweet = re.sub(r"mortalkombat", "Mortal Kombat", tweet)
    tweet = re.sub(r"FilipeCoelho92", "Filipe Coelho", tweet)
    tweet = re.sub(r"OnlyQuakeNews", "Only Quake News", tweet)
    tweet = re.sub(r"kostumes", "costumes", tweet)
    tweet = re.sub(r"YEEESSSS", "yes", tweet)
    tweet = re.sub(r"ToshikazuKatayama", "Toshikazu Katayama", tweet)
    tweet = re.sub(r"IntlDevelopment", "Intl Development", tweet)
    tweet = re.sub(r"ExtremeWeather", "Extreme Weather", tweet)
    tweet = re.sub(r"WereNotGruberVoters", "We are not gruber voters", tweet)
    tweet = re.sub(r"NewsThousands", "News Thousands", tweet)
    tweet = re.sub(r"EdmundAdamus", "Edmund Adamus", tweet)
    tweet = re.sub(r"EyewitnessWV", "Eye witness WV", tweet)
    tweet = re.sub(r"PhiladelphiaMuseu", "Philadelphia Museum", tweet)
    tweet = re.sub(r"DublinComicCon", "Dublin Comic Con", tweet)
    tweet = re.sub(r"NicholasBrendon", "Nicholas Brendon", tweet)
    tweet = re.sub(r"Alltheway80s", "All the way 80s", tweet)
    tweet = re.sub(r"FromTheField", "From the field", tweet)
    tweet = re.sub(r"NorthIowa", "North Iowa", tweet)
    tweet = re.sub(r"WillowFire", "Willow Fire", tweet)
    tweet = re.sub(r"MadRiverComplex", "Mad River Complex", tweet)
    tweet = re.sub(r"feelingmanly", "feeling manly", tweet)
    tweet = re.sub(r"stillnotoverit", "still not over it", tweet)
    tweet = re.sub(r"FortitudeValley", "Fortitude Valley", tweet)
    tweet = re.sub(r"CoastpowerlineTramTr", "Coast powerline", tweet)
    tweet = re.sub(r"ServicesGold", "Services Gold", tweet)
    tweet = re.sub(r"NewsbrokenEmergency", "News broken emergency", tweet)
    tweet = re.sub(r"Evaucation", "evacuation", tweet)
    tweet = re.sub(r"leaveevacuateexitbe", "leave evacuate exit be", tweet)
    tweet = re.sub(r"P_EOPLE", "PEOPLE", tweet)
    tweet = re.sub(r"Tubestrike", "tube strike", tweet)
    tweet = re.sub(r"CLASS_SICK", "CLASS SICK", tweet)
    tweet = re.sub(r"localplumber", "local plumber", tweet)
    tweet = re.sub(r"awesomejobsiri", "awesome job siri", tweet)
    tweet = re.sub(r"PayForItHow", "Pay for it how", tweet)
    tweet = re.sub(r"ThisIsAfrica", "This is Africa", tweet)
    tweet = re.sub(r"crimeairnetwork", "crime air network", tweet)
    tweet = re.sub(r"KimAcheson", "Kim Acheson", tweet)
    tweet = re.sub(r"cityofcalgary", "City of Calgary", tweet)
    tweet = re.sub(r"prosyndicate", "pro syndicate", tweet)
    tweet = re.sub(r"660NEWS", "660 NEWS", tweet)
    tweet = re.sub(r"BusInsMagazine", "Business Insurance Magazine", tweet)
    tweet = re.sub(r"wfocus", "focus", tweet)
    tweet = re.sub(r"ShastaDam", "Shasta Dam", tweet)
    tweet = re.sub(r"go2MarkFranco", "Mark Franco", tweet)
    tweet = re.sub(r"StephGHinojosa", "Steph Hinojosa", tweet)
    tweet = re.sub(r"Nashgrier", "Nash Grier", tweet)
    tweet = re.sub(r"NashNewVideo", "Nash new video", tweet)
    tweet = re.sub(r"IWouldntGetElectedBecause", "I would not get elected because", tweet)
    tweet = re.sub(r"SHGames", "Sledgehammer Games", tweet)
    tweet = re.sub(r"bedhair", "bed hair", tweet)
    tweet = re.sub(r"JoelHeyman", "Joel Heyman", tweet)
    tweet = re.sub(r"viaYouTube", "via YouTube", tweet)

    # URLs t.co fallback
    tweet = re.sub(r"https?:\/\/t.co\/[A-Za-z0-9]+", "", tweet)

    # Ponctuation
    punctuations = '@#!?+&*[]-%.:/();$=><|{}^' + "'\`"
    for p in punctuations:
        tweet = tweet.replace(p, f' {p} ')

    tweet = tweet.replace('...', ' ... ')
    if '...' not in tweet:
        tweet = tweet.replace('..', ' ... ')

    # ── ACRONYMES ───────────────────────────────────────────────────────────
    tweet = re.sub(r"MH370", "Malaysia Airlines Flight 370", tweet)
    tweet = re.sub(r"mÌ¼sica", "music", tweet)
    tweet = re.sub(r"okwx", "Oklahoma City Weather", tweet)
    tweet = re.sub(r"arwx", "Arkansas Weather", tweet)
    tweet = re.sub(r"gawx", "Georgia Weather", tweet)
    tweet = re.sub(r"scwx", "South Carolina Weather", tweet)
    tweet = re.sub(r"cawx", "California Weather", tweet)
    tweet = re.sub(r"tnwx", "Tennessee Weather", tweet)
    tweet = re.sub(r"azwx", "Arizona Weather", tweet)
    tweet = re.sub(r"alwx", "Alabama Weather", tweet)
    tweet = re.sub(r"wordpressdotcom", "wordpress", tweet)
    tweet = re.sub(r"usNWSgov", "United States National Weather Service", tweet)
    tweet = re.sub(r"Suruc", "Sanliurfa", tweet)

    # Grouping
    tweet = re.sub(r"Bestnaijamade", "bestnaijamade", tweet)
    tweet = re.sub(r"SOUDELOR", "Soudelor", tweet)

    # Normalisation finale
    tweet = re.sub(r'\s+', ' ', tweet).strip()
    return tweet


def basic_clean(text):
    """Nettoyage générique : lowercase, emojis, URLs, mentions, hashtags, ponctuation."""
    text = str(text).lower()
    text = emoji.demojize(text, delimiters=(' ', ' '))
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def tokenize_no_stop(text):
    tokens = basic_clean(text).split()
    return [t for t in tokens if t not in STOPWORDS_EN and len(t) > 2]


def full_clean(tweet):
    """Pipeline complet : clean() cas par cas → basic_clean() générique."""
    tweet = clean(tweet)
    tweet = basic_clean(tweet)
    return tweet


def clean_location(location):
    if pd.isna(location) or str(location).lower() == 'unknown':
        return 'unknown'
    location = str(location).lower()
    location = re.sub(r'[^a-z0-9\s]', ' ', location)
    location = re.sub(r'\s+', ' ', location).strip()
    location = re.sub(r'\bnew york city\b', 'new york city', location)
    location = re.sub(r'\bnew york\b', 'new york city', location)
    location = re.sub(r'\bnyc\b', 'new york city', location)
    location = re.sub(r'\bunited states\b', 'usa', location)
    location = re.sub(r'\bus\b', 'usa', location)
    location = re.sub(r'\bunited kingdom\b', 'uk', location)
    location = re.sub(r'\blondon england\b', 'london uk', location)
    location = re.sub(r'\blos angeles california\b', 'los angeles california', location)
    location = re.sub(r'\blos angeles ca\b', 'los angeles california', location)
    location = re.sub(r'\bla\b', 'los angeles california', location)
    location = re.sub(r'\bcalifornia usa\b', 'california', location)
    location = re.sub(r'\bwashington dc\b', 'washington', location)
    location = re.sub(r'\bwashington state\b', 'washington', location)
    if not location:
        return 'unknown'
    return location


def clean_keyword(keyword):
    if pd.isna(keyword):
        return 'unknown'
    keyword = str(keyword).replace('%20', ' ').lower()
    keyword = re.sub(r'[^a-z0-9\s]', '', keyword)
    keyword = re.sub(r'\s+', ' ', keyword).strip()
    return keyword

print('✅ Fonctions de nettoyage définies')

### 4.2 Application & Split train/test

In [ ]:
print('Nettoyage en cours...')
df['clean_text']     = df['text'].apply(full_clean)
df['tokens']         = df['text'].apply(tokenize_no_stop)
df['clean_location'] = df['location'].apply(clean_location)
df['clean_location'] = df['clean_location'].fillna('unknown')
df['clean_keyword']  = df['keyword'].apply(clean_keyword)

train, test = train_test_split(
    df,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df['target']
)
train = train.reset_index(drop=True)
test  = test.reset_index(drop=True)

print(f'Train shape : {train.shape}')
print(f'Test  shape : {test.shape}')
train.head()

### 4.3 Vérification du nettoyage — exemples

In [ ]:
# Exemples de tweets avant / après nettoyage
print('=== Exemples avant/après nettoyage ===')
display(train[['text', 'clean_text', 'target']].sample(5, random_state=RANDOM_STATE))

# Test emoji
sample_emoji = "This is so sad 😭 but also amazing! ✨ ambulance 🚑"
print(f'\nOriginal : {sample_emoji}')
print(f'Nettoyé  : {full_clean(sample_emoji)}')

## 5. Keyword & Location

### 5.1 Keyword

In [ ]:
print(f"Keywords uniques (train) : {train['keyword'].nunique()}")
print(f"Keywords uniques (test)  : {test['keyword'].nunique()}")

top_kw = train['keyword'].value_counts().head(20)

fig, ax = plt.subplots(figsize=(12, 7))
top_kw.sort_values().plot(kind='barh', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Top 20 Keywords les plus fréquents (train)', fontsize=13, fontweight='bold')
ax.set_xlabel('Fréquence')
plt.tight_layout()
plt.show()

In [ ]:
# Taux de disaster par keyword (top 20)
kw_stats = (
    train.groupby('keyword')['target']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'disaster_rate', 'count': 'total'})
    .sort_values('total', ascending=False)
    .head(20)
)

fig, ax = plt.subplots(figsize=(14, 7))
ax.barh(kw_stats.index, kw_stats['disaster_rate'],
        color=['#FF6B6B' if r > 0.5 else '#4ECDC4' for r in kw_stats['disaster_rate']],
        edgecolor='black')
ax.axvline(0.5, color='black', linestyle='--', linewidth=1.5, label='Seuil 50%')
ax.set_xlabel('Taux de disaster')
ax.set_title('Taux de "disaster" par keyword (Top 20 plus fréquents)', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Impact de l'absence de keyword sur la cible
train['has_keyword'] = train['keyword'].notna().astype(int)
kw_missing_rate = train.groupby('has_keyword')['target'].mean()

print('Taux de disaster selon présence du keyword :')
print(kw_missing_rate.rename({0: 'Keyword absent', 1: 'Keyword présent'}))

fig, ax = plt.subplots(figsize=(6, 4))
kw_missing_rate.plot(kind='bar', ax=ax, color=['#FF6B6B', '#4ECDC4'], edgecolor='black', width=0.4)
ax.set_xticklabels(['Keyword absent', 'Keyword présent'], rotation=0)
ax.set_ylabel('Taux de disaster')
ax.set_title('Taux de disaster : keyword présent vs absent', fontsize=12, fontweight='bold')
ax.set_ylim(0, 1)
for i, v in enumerate(kw_missing_rate.values):
    ax.text(i, v + 0.02, f'{v:.2f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

### 5.2 Location

In [ ]:
print(f"Locations uniques (train) : {train['location'].nunique()}")

top_loc = train['location'].value_counts().head(20)

fig, ax = plt.subplots(figsize=(12, 7))
top_loc.sort_values().plot(kind='barh', ax=ax, color='mediumpurple', edgecolor='black')
ax.set_title('Top 20 Locations (train)', fontsize=13, fontweight='bold')
ax.set_xlabel('Fréquence')
plt.tight_layout()
plt.show()

## 6. Features textuelles

In [ ]:
def extract_text_features(dataframe):
    dataframe = dataframe.copy()
    dataframe['char_count']        = dataframe['text'].str.len()
    dataframe['word_count']        = dataframe['text'].str.split().str.len()
    dataframe['unique_word_count'] = dataframe['text'].apply(lambda x: len(set(str(x).split())))
    dataframe['avg_word_length']   = dataframe['text'].apply(
        lambda x: np.mean([len(w) for w in str(x).split()]) if len(str(x).split()) > 0 else 0
    )
    dataframe['url_count']         = dataframe['text'].apply(lambda x: len(re.findall(r'http\S+', str(x))))
    dataframe['mention_count']     = dataframe['text'].apply(lambda x: len(re.findall(r'@\w+', str(x))))
    dataframe['hashtag_count']     = dataframe['text'].apply(lambda x: len(re.findall(r'#\w+', str(x))))
    dataframe['exclamation_count'] = dataframe['text'].str.count('!')
    dataframe['question_count']    = dataframe['text'].str.count('\\?')
    dataframe['digit_count']       = dataframe['text'].apply(lambda x: sum(c.isdigit() for c in str(x)))
    dataframe['uppercase_count']   = dataframe['text'].apply(lambda x: sum(c.isupper() for c in str(x)))
    return dataframe

train = extract_text_features(train)
test  = extract_text_features(test)
print('✅ Features textuelles ajoutées')
train.head(2)

In [ ]:
disaster     = train[train['target'] == 1]
non_disaster = train[train['target'] == 0]

features_to_plot = [
    ('char_count',      'Nombre de caractères'),
    ('word_count',      'Nombre de mots'),
    ('avg_word_length', 'Longueur moyenne des mots'),
    ('url_count',       'Nombre de URLs'),
    ('mention_count',   'Nombre de mentions (@)'),
    ('hashtag_count',   'Nombre de hashtags (#)'),
]

fig, axes = plt.subplots(3, 2, figsize=(14, 12))
axes = axes.flatten()

for i, (feat, label) in enumerate(features_to_plot):
    axes[i].hist(disaster[feat],     bins=30, alpha=0.6, color='#FF6B6B', label='Disaster (1)',     density=True)
    axes[i].hist(non_disaster[feat], bins=30, alpha=0.6, color='#4ECDC4', label='Non-Disaster (0)', density=True)
    axes[i].set_title(label, fontsize=11, fontweight='bold')
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('Densité')
    axes[i].legend()

plt.suptitle('Distribution des features textuelles par classe', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 6.1 Matrice de corrélation

In [ ]:
num_features = [
    'char_count', 'word_count', 'unique_word_count',
    'avg_word_length', 'url_count', 'mention_count',
    'hashtag_count', 'exclamation_count', 'question_count',
    'digit_count', 'uppercase_count', 'target'
]

corr_matrix = train[num_features].corr()

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, ax=ax,
    linewidths=0.5, square=True, cbar_kws={'shrink': 0.8}
)
ax.set_title('Matrice de corrélation des features numériques', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Analyse textuelle

### 7.1 Fréquences de mots par classe

In [ ]:
# Tokens par classe (recalcul sur train splitté)
disaster     = train[train['target'] == 1]
non_disaster = train[train['target'] == 0]

all_tokens          = [t for tokens in train['tokens']        for t in tokens]
tokens_disaster     = [t for tokens in disaster['tokens']     for t in tokens]
tokens_non_disaster = [t for tokens in non_disaster['tokens'] for t in tokens]

word_freq         = Counter(all_tokens)
freq_disaster     = Counter(tokens_disaster)
freq_non_disaster = Counter(tokens_non_disaster)

print(f'Vocabulaire total (sans stopwords) : {len(word_freq)}')
print(f'Tokens disaster     : {len(tokens_disaster)} ({len(freq_disaster)} uniques)')
print(f'Tokens non-disaster : {len(tokens_non_disaster)} ({len(freq_non_disaster)} uniques)')

In [ ]:
top30 = pd.DataFrame(word_freq.most_common(30), columns=['word', 'count'])

fig, ax = plt.subplots(figsize=(14, 7))
ax.barh(top30['word'][::-1], top30['count'][::-1], color='steelblue', edgecolor='black')
ax.set_title('Top 30 mots les plus fréquents (sans stopwords) — Corpus complet', fontsize=13, fontweight='bold')
ax.set_xlabel('Fréquence')
plt.tight_layout()
plt.show()

In [ ]:
top_dis  = pd.DataFrame(freq_disaster.most_common(20),     columns=['word', 'count'])
top_ndis = pd.DataFrame(freq_non_disaster.most_common(20), columns=['word', 'count'])

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

axes[0].barh(top_dis['word'][::-1], top_dis['count'][::-1], color='#FF6B6B', edgecolor='black')
axes[0].set_title('Top 20 mots — Disaster (1)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Fréquence')

axes[1].barh(top_ndis['word'][::-1], top_ndis['count'][::-1], color='#4ECDC4', edgecolor='black')
axes[1].set_title('Top 20 mots — Non-Disaster (0)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Fréquence')

plt.suptitle('Top 20 mots les plus fréquents par classe (sans stopwords)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 7.2 Lexical Diversity (Type-Token Ratio)

In [ ]:
# Type-Token Ratio : tokens uniques / tokens totaux
# Un ratio élevé = vocabulaire varié ; faible = vocabulaire répétitif
def ttr(token_list):
    if len(token_list) == 0:
        return 0
    return len(set(token_list)) / len(token_list)

train['ttr'] = train['tokens'].apply(ttr)
disaster     = train[train['target'] == 1]
non_disaster = train[train['target'] == 0]

print(f'TTR moyen — Disaster     : {disaster["ttr"].mean():.4f}')
print(f'TTR moyen — Non-Disaster : {non_disaster["ttr"].mean():.4f}')

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(disaster['ttr'],     bins=30, alpha=0.6, color='#FF6B6B', label='Disaster (1)',     density=True)
ax.hist(non_disaster['ttr'], bins=30, alpha=0.6, color='#4ECDC4', label='Non-Disaster (0)', density=True)
ax.set_title('Type-Token Ratio par classe', fontsize=13, fontweight='bold')
ax.set_xlabel('TTR')
ax.set_ylabel('Densité')
ax.legend()
plt.tight_layout()
plt.show()

### 7.3 Longueur des tweets avant / après nettoyage

In [ ]:
train['clean_word_count'] = train['clean_text'].str.split().str.len()
train['tokens_lost']      = train['word_count'] - train['clean_word_count']

print('Tokens perdus après nettoyage (moyenne par classe) :')
print(train.groupby('target')[['word_count', 'clean_word_count', 'tokens_lost']].mean().round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, title in zip(axes,
                           ['word_count', 'clean_word_count'],
                           ['Avant nettoyage', 'Après nettoyage']):
    ax.hist(disaster[col],     bins=30, alpha=0.6, color='#FF6B6B', label='Disaster',     density=True)
    ax.hist(non_disaster[col], bins=30, alpha=0.6, color='#4ECDC4', label='Non-Disaster', density=True)
    ax.set_title(f'Nb de mots — {title}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Nombre de mots')
    ax.set_ylabel('Densité')
    ax.legend()

plt.suptitle('Longueur des tweets : avant vs après nettoyage', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 7.4 Tweets trop courts après nettoyage

In [ ]:
# Tweets avec moins de 3 tokens après nettoyage → risque pour la modélisation
short_tweets = train[train['clean_word_count'] < 3]
print(f'Tweets avec < 3 mots après nettoyage : {len(short_tweets)} ({len(short_tweets)/len(train)*100:.1f}%)')
print(f'  dont Disaster     : {short_tweets["target"].sum()}')
print(f'  dont Non-Disaster : {(short_tweets["target"] == 0).sum()}')
print()
print('Exemples :')
display(short_tweets[['text', 'clean_text', 'target']].head(10))

## 8. WordClouds par classe

In [ ]:
def make_wordcloud(word_dict, background_color='white', colormap='Blues'):
    wc = WordCloud(
        background_color=background_color,
        max_words=200,
        colormap=colormap,
        collocations=False,
        width=800, height=400
    ).generate_from_frequencies(word_dict)
    return wc

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

wc_all  = make_wordcloud(dict(word_freq),         colormap='Blues')
wc_dis  = make_wordcloud(dict(freq_disaster),     colormap='Reds')
wc_ndis = make_wordcloud(dict(freq_non_disaster), colormap='Greens')

for ax, wc, title in zip(axes,
                          [wc_all, wc_dis, wc_ndis],
                          ['Corpus global', 'Disaster (target=1)', 'Non-Disaster (target=0)']):
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.axis('off')

plt.suptitle('WordClouds par classe', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. N-grammes

### 9.1 Bigrammes

In [ ]:
def get_ngram_freq(token_series, n=2, top_k=20):
    all_ngrams = []
    for tokens in token_series:
        all_ngrams.extend(list(ngrams(tokens, n)))
    ngram_freq = Counter(all_ngrams)
    top = pd.DataFrame(
        [(' '.join(gram), count) for gram, count in ngram_freq.most_common(top_k)],
        columns=['ngram', 'count']
    )
    return top

bi_dis  = get_ngram_freq(disaster['tokens'],     n=2)
bi_ndis = get_ngram_freq(non_disaster['tokens'], n=2)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

axes[0].barh(bi_dis['ngram'][::-1],  bi_dis['count'][::-1],  color='#FF6B6B', edgecolor='black')
axes[0].set_title('Top 20 Bigrammes — Disaster (1)',     fontsize=12, fontweight='bold')
axes[0].set_xlabel('Fréquence')

axes[1].barh(bi_ndis['ngram'][::-1], bi_ndis['count'][::-1], color='#4ECDC4', edgecolor='black')
axes[1].set_title('Top 20 Bigrammes — Non-Disaster (0)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Fréquence')

plt.suptitle('Top 20 Bigrammes par classe', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 9.2 Trigrammes

In [ ]:
tri_dis  = get_ngram_freq(disaster['tokens'],     n=3)
tri_ndis = get_ngram_freq(non_disaster['tokens'], n=3)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

axes[0].barh(tri_dis['ngram'][::-1],  tri_dis['count'][::-1],  color='#FF6B6B', edgecolor='black')
axes[0].set_title('Top 20 Trigrammes — Disaster (1)',     fontsize=12, fontweight='bold')
axes[0].set_xlabel('Fréquence')

axes[1].barh(tri_ndis['ngram'][::-1], tri_ndis['count'][::-1], color='#4ECDC4', edgecolor='black')
axes[1].set_title('Top 20 Trigrammes — Non-Disaster (0)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Fréquence')

plt.suptitle('Top 20 Trigrammes par classe', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. TF-IDF — Mots les plus discriminants

In [ ]:
# TF-IDF sur le corpus complet — top mots par classe
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=3)
X_tfidf = tfidf.fit_transform(train['clean_text'])

feature_names = np.array(tfidf.get_feature_names_out())

# Moyenne TF-IDF par classe
mean_dis  = X_tfidf[train['target'] == 1].mean(axis=0).A1
mean_ndis = X_tfidf[train['target'] == 0].mean(axis=0).A1

# Top 20 par classe
top20_dis  = pd.DataFrame({'word': feature_names, 'tfidf': mean_dis}).nlargest(20, 'tfidf')
top20_ndis = pd.DataFrame({'word': feature_names, 'tfidf': mean_ndis}).nlargest(20, 'tfidf')

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

axes[0].barh(top20_dis['word'][::-1],  top20_dis['tfidf'][::-1],  color='#FF6B6B', edgecolor='black')
axes[0].set_title('Top 20 TF-IDF — Disaster (1)',     fontsize=12, fontweight='bold')
axes[0].set_xlabel('Score TF-IDF moyen')

axes[1].barh(top20_ndis['word'][::-1], top20_ndis['tfidf'][::-1], color='#4ECDC4', edgecolor='black')
axes[1].set_title('Top 20 TF-IDF — Non-Disaster (0)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Score TF-IDF moyen')

plt.suptitle('Mots les plus discriminants par classe (TF-IDF)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Score de différenciation : mots avec le plus grand écart TF-IDF entre classes
diff_df = pd.DataFrame({
    'word'       : feature_names,
    'tfidf_dis'  : mean_dis,
    'tfidf_ndis' : mean_ndis,
})
diff_df['diff'] = diff_df['tfidf_dis'] - diff_df['tfidf_ndis']

top_pro_dis  = diff_df.nlargest(15, 'diff')
top_pro_ndis = diff_df.nsmallest(15, 'diff')

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

axes[0].barh(top_pro_dis['word'][::-1],  top_pro_dis['diff'][::-1],  color='#FF6B6B', edgecolor='black')
axes[0].set_title('Mots les plus pro-Disaster',     fontsize=12, fontweight='bold')
axes[0].set_xlabel('Différence TF-IDF (Disaster - Non-Disaster)')

axes[1].barh(top_pro_ndis['word'][::-1], top_pro_ndis['diff'].abs()[::-1], color='#4ECDC4', edgecolor='black')
axes[1].set_title('Mots les plus pro-Non-Disaster', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Différence TF-IDF (Non-Disaster - Disaster)')

plt.suptitle('Mots les plus différenciants entre classes', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 11. Hashtags / Mentions / URLs

In [ ]:
def pct_with(series, min_val=1):
    return (series >= min_val).mean() * 100

print('=== Tweets contenant au moins un pattern spécial ===')
for feat in ['url_count', 'mention_count', 'hashtag_count']:
    total = pct_with(train[feat])
    dis   = pct_with(disaster[feat])
    ndis  = pct_with(non_disaster[feat])
    print(f'{feat:20s}  Total: {total:.1f}%  |  Disaster: {dis:.1f}%  |  Non-Disaster: {ndis:.1f}%')

In [ ]:
def extract_hashtags(text):
    return re.findall(r'#(\w+)', str(text).lower())

hashtags_dis  = [h for text in disaster['text']     for h in extract_hashtags(text)]
hashtags_ndis = [h for text in non_disaster['text'] for h in extract_hashtags(text)]

top_ht_dis  = Counter(hashtags_dis).most_common(15)
top_ht_ndis = Counter(hashtags_ndis).most_common(15)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

if top_ht_dis:
    words_d, counts_d = zip(*top_ht_dis)
    axes[0].barh(words_d[::-1], counts_d[::-1], color='#FF6B6B', edgecolor='black')
    axes[0].set_title('Top Hashtags — Disaster', fontsize=12, fontweight='bold')

if top_ht_ndis:
    words_n, counts_n = zip(*top_ht_ndis)
    axes[1].barh(words_n[::-1], counts_n[::-1], color='#4ECDC4', edgecolor='black')
    axes[1].set_title('Top Hashtags — Non-Disaster', fontsize=12, fontweight='bold')

plt.suptitle('Hashtags les plus fréquents par classe', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 12. Doublons & Labels conflictuels

In [ ]:
dup_text = train[train.duplicated('text', keep=False)]
print(f'Tweets en doublon (texte identique) : {len(dup_text)} ({len(dup_text)/len(train)*100:.1f}%)')

dup_conflict = (
    train.groupby('text')['target']
    .nunique()
    .reset_index()
    .query('target > 1')
)
print(f'Tweets avec labels conflictuels : {len(dup_conflict)}')

if len(dup_conflict) > 0:
    print(f'\n⚠️ Ces {len(dup_conflict)} tweets apparaissent avec les deux labels (0 et 1).')
    print('Ils constituent du bruit pour la modélisation.\n')
    print('Exemples de tweets ambigus :')
    conflict_examples = train[train['text'].isin(dup_conflict['text'].head(5))]
    display(conflict_examples[['text', 'target']].sort_values('text'))

## 13. Train vs Test

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, dataframe, title in zip(axes, [train, test], ['Train', 'Test']):
    mp = dataframe.isnull().mean() * 100
    mp = mp[mp > 0]
    if len(mp) > 0:
        ax.barh(mp.index, mp.values, color='#FF6B6B', edgecolor='black')
        ax.set_xlabel('% manquant')
        ax.set_title(f'Valeurs manquantes — {title}', fontsize=13, fontweight='bold')
        for i, v in enumerate(mp.values):
            ax.text(v + 0.3, i, f'{v:.1f}%', va='center')
    else:
        ax.text(0.5, 0.5, 'Aucune valeur manquante', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'Valeurs manquantes — {title}', fontsize=13)

plt.tight_layout()
plt.show()

In [ ]:
# Comparaison distribution des features textuelles train vs test
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, feat in zip(axes, ['char_count', 'word_count', 'hashtag_count', 'url_count']):
    ax.hist(train[feat], bins=30, alpha=0.6, color='steelblue', label='Train', density=True)
    ax.hist(test[feat],  bins=30, alpha=0.6, color='orange',    label='Test',  density=True)
    ax.set_title(feat, fontsize=11, fontweight='bold')
    ax.legend()

plt.suptitle('Distribution des features : Train vs Test', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 14. Longueur de séquence (préparation modèles)

In [ ]:
# Estimation de la longueur en tokens (split naïf — approximation avant tokenisation BERT)
# Pour une tokenisation WordPiece exacte, utiliser transformers.AutoTokenizer
train['token_len'] = train['clean_text'].apply(lambda x: len(str(x).split()))

print('Statistiques de longueur (tokens, après nettoyage) :')
print(train['token_len'].describe().round(1))
print()
print(f'% tweets <= 50 tokens  : {(train["token_len"] <= 50).mean()*100:.1f}%')
print(f'% tweets <= 100 tokens : {(train["token_len"] <= 100).mean()*100:.1f}%')
print(f'% tweets <= 128 tokens : {(train["token_len"] <= 128).mean()*100:.1f}%')
print(f'% tweets >  128 tokens : {(train["token_len"] > 128).mean()*100:.1f}%  ← tronqués par BERT base')

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(train['token_len'], bins=50, color='steelblue', edgecolor='black', alpha=0.8)
ax.axvline(128, color='red',    linestyle='--', linewidth=2, label='Limite BERT (128)')
ax.axvline(512, color='orange', linestyle='--', linewidth=2, label='Limite BERT max (512)')
ax.set_title('Distribution de la longueur des tweets (tokens)', fontsize=13, fontweight='bold')
ax.set_xlabel('Nombre de tokens')
ax.set_ylabel('Nombre de tweets')
ax.legend()
plt.tight_layout()
plt.show()

## 15. Exemples de tweets par classe

In [ ]:
print('=== Exemples DISASTER (target=1) ===')
for i, row in disaster[['text', 'keyword']].sample(5, random_state=42).iterrows():
    print(f'[{i}] {row["text"]}')
    print(f'    keyword: {row["keyword"]}')
    print()

print('=== Exemples NON-DISASTER (target=0) ===')
for i, row in non_disaster[['text', 'keyword']].sample(5, random_state=42).iterrows():
    print(f'[{i}] {row["text"]}')
    print(f'    keyword: {row["keyword"]}')
    print()

## 16. Résumé & Insights

### Ce qu'on a appris sur le dataset

**Dataset & cible**
- Déséquilibre modéré : ~57% Non-Disaster / ~43% Disaster → pas besoin d'oversampling agressif mais à surveiller.
- Les tweets sans keyword ont un taux de disaster différent → la présence du keyword est un signal utile.

**Features textuelles**
- Les tweets disaster sont légèrement plus longs (plus de caractères, plus de mots).
- Le nombre de majuscules est légèrement plus élevé dans les tweets disaster (urgence, cris).
- Les URLs et mentions sont plus fréquentes dans les tweets non-disaster (contenu promotionnel, divertissement).

**Vocabulaire**
- Les mots disaster (fire, flood, storm, killed, people, dead...) sont clairement séparés des mots non-disaster.
- Le Type-Token Ratio révèle que les tweets disaster ont un vocabulaire légèrement plus répétitif (termes techniques récurrents).
- Les bigrammes et trigrammes confirment cette séparation : "suicide bomber", "body bags" vs "birthday", "happy", "love".

**Qualité des données**
- Des doublons avec labels conflictuels existent → bruit à gérer (supprimer ou conserver le label majoritaire).
- Quelques tweets deviennent très courts après nettoyage → à surveiller pour les modèles sensibles à la longueur.
- La longueur est quasi-entièrement sous 128 tokens → BERT base avec max_length=128 convient parfaitement.

**Pipeline de nettoyage**
- `clean()` : remplacements cas par cas (contractions, encodage, hashtags nommés, typos)
- `basic_clean()` : nettoyage générique (lowercase, emojis→texte, URLs, mentions, ponctuation)
- `full_clean()` : les deux en séquence → colonne `clean_text` prête pour la modélisation

## 17. Sauvegarde des données

In [ ]:
# Export des datasets nettoyés pour la modélisation
train.to_csv('train_cleaned.csv', index=False)
test.to_csv('test_cleaned.csv',   index=False)
print('✅ Fichiers train_cleaned.csv et test_cleaned.csv sauvegardés !')
print(f'   train : {train.shape}')
print(f'   test  : {test.shape}')
print(f'   colonnes : {list(train.columns)}')